In [1]:
import os, shutil
import pandas as pd, numpy as np
# sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')

from FotR import GANDALF


[flexo.service:07120] pml_ucx.c:313  Error: Failed to create UCP worker
0 Warning! Import - NVTX not present!


[1779752076.978545] [flexo:7120 :0]        ib_iface.c:1317 UCX  ERROR mlx5_1: ibv_create_cq(cqe=4096) failed: Cannot allocate memory : Please set max locked memory (ulimit -l) to 'unlimited' (current: 64 kbytes)
[1779752076.978580] [flexo:7120 :0]      ucp_worker.c:1412 UCX  ERROR uct_iface_open(rc_verbs/mlx5_1:1) failed: Input/output error


# CODA 2025

In [2]:
try:
    shutil.rmtree("/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/")
except:
    print('No la encuentro')
    pass
# import pandas as pd
# df_post = pd.read_csv('/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22/rans3/metadata/df_post.csv', sep=',', usecols=['aoa', 'mach', 'h', 'case_idx'])

No la encuentro


In [3]:

# EAREN = EarendilsLight(FRODO)

# EAREN.help("CODAReader")

dataset_dir = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON'

gimli = GANDALF(
    os.path.join(dataset_dir, "rans3_transonic_1"),
    eq_type="rans", 
    num_stages = 2,
    version="flowsimulator2024"
)
# cases = [64, 79, 87, 88, 94]
# gimli.define_cases(
#     method="external",
#     external_dataframe = pd.DataFrame({ # Caso 79 de df_post.csv
#         "AoA": df_post['aoa'].loc[cases],
#         "Mach": df_post['mach'].loc[cases],
#         "h": df_post['h'].loc[cases]
#     }),
# )

gimli.define_cases(
    method="Halton",
    bounds={"AoA": (0.0, 5.0), "Mach": (0.8, 1.1)}, # {"AoA": (0.0, 5.0), "Mach": (0.3, 1.5), 'h': 11000},
    n_samples=20,
    peak_ranges=None,
    seed=42,
)


New simulation folder will be created in /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1.


In [4]:
geom_csv = 'eta_0_0.csv'

gimli.define_geom_file(
    geom_file_path=os.path.join(dataset_dir, 'airfoil_files', geom_csv), sep=',', decimal='.', header = 'infer',
    cols_idx=[1, 3],
    normalize = False
)

c = gimli.array_ptos[:, 0].max() - gimli.array_ptos[:, 0].min()
cm = -c/4 # la malla lo tiene ya ajustado a x=0 el borde de ataque


In [5]:
print(gimli.df_cases.columns)
T, P, rho = GANDALF.Backpack.isa_atmosphere(h=11000)
mu = GANDALF.Backpack.Sutherland_law(mu0 = 1.716e-5, T=T, Treference=255.55)

gimli.compute_param(
    name = 'Re',
    formula = "L * Mach*sqrt(1.4*287*T)*rho/mu",
    externals={
        "rho": rho,
        "mu": mu,
        "L": c,
        "T": T
    }
)
print(gimli.df_cases.columns)

Index(['AoA', 'Mach'], dtype='object')
Index(['AoA', 'Mach', 'Re'], dtype='object')


In [6]:
gimli.generate_folders(
    base_files = ["run_sst_v4.py", "run.sh"],
    mesh_path=os.path.join(dataset_dir, "sources", "mesh_f22_transonic.msh"),
    script_dir=os.path.join(dataset_dir, "sources"),
    folder_fmt = "aoa_{AoA:.4f}_mach_{Mach:.4f}",
    overwrite=True,
    update_base_files=True,
    data_to_update={
        "AOA_PLACEHOLDER": "AoA",
        "MACH_PLACEHOLDER": "Mach",
        # "ALT_PLACEHOLDER": "h",
        "PYTHON_FILE_PLACEHOLDER": "run_sst_v4.py",
        "CHORD_PLACEHOLDER": str(c),
        "CM_PLACEHOLDER": str(cm)
    }
)

Creating 20 simulation folders in /home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1


In [7]:
# df_sacar = gimli.df_cases.copy()
# df_sacar['AoA'] = pd.to_numeric(df_sacar['AoA'])
# df_sacar['Mach'] = pd.to_numeric(df_sacar['Mach'])
# df_definitivo = df_sacar.round({'AoA': 4, 'Mach': 4})
# df_definitivo['Coef_Area'] = c
# df_definitivo.to_csv(os.path.join(gimli.root_dir,'metadata', 'df_cases.csv'), index=False)
gimli.df_cases['Coef_Area'] = c
gimli.df_cases.to_csv(os.path.join(gimli.root_dir,'metadata', 'df_cases.csv'), index=False)

In [10]:
gimli.assign_jobs(
    file_sh="run.sh",
    nodes=[f'n00{n}' for n in [3, 4, 5]],
    cpus_per_job=48,
    submit=True,
    )


--- Trabajos en ejecución ---
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             41221       gpu     1_PL   p.sole  R 4-13:47:20      1 n008
             41280       gpu     2_PL   p.sole  R 2-13:56:41      1 n008
             41353       gpu     2_PL   p.sole  R 1-06:31:43      1 n008
             41357       gpu   1_FULL   p.sole  R   15:15:47      1 n008
             41358    normal caso11_f   g.cano  R   13:34:03      1 n001
             41359    normal caso11_g   g.cano  R   13:33:59      1 n001
             41360    normal caso11_m   g.cano  R   13:33:56      1 n001
             41361    normal caso12_f   g.cano  R   13:33:51      1 n001
             41362    normal caso12_g   g.cano  R   13:33:48      1 n001
             41363    normal caso12_m   g.cano  R   13:33:45      1 n001
             41364    normal caso11_g   g.cano  R   13:33:35      1 n001
             41365    normal caso11_m   g.cano  R   13:33:30      1 n001
        

In [11]:
gimli.submit_cases()


--- Submitting jobs ---

/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_2.7565_mach_0.8455/run.sh
aoa_2.7565_mach_0.8455: Submitted batch job 41388
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_0.2565_mach_1.0455/run.sh
aoa_0.2565_mach_1.0455: Submitted batch job 41389
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_4.0065_mach_0.9455/run.sh
aoa_4.0065_mach_0.9455: Submitted batch job 41390
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_1.5065_mach_0.8789/run.sh
aoa_1.5065_mach_0.8789: Submitted batch job 41391
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_3.3815_mach_1.0789/run.sh
aoa_3.3815_mach_1.0789: Submitted batch job 41392
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_1/outputs/aoa_0.8815_mach_0.9789/run.sh
aoa_0.8815_mach_0.9789: Submitted batch job 41393
/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_transonic_

# FLUENT R22

In [ ]:
try:
    shutil.rmtree("./f22_fluent/outputs")
except:
    pass

gimli = GIMLI("./f22_fluent/", eq_type="rans")

gimli.define_cases(
    method="Halton",
    bounds={"AoA": (0.0, 5.0), "Mach": (0.3, 1.5), 'h': 11000},
    n_samples=5,
    peak_ranges=None,
    seed=42,
)

dataset_dir = "/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22/airfoil_files/"
geom_csv = 'eta_0_0.csv'

gimli.define_geom_file(
    geom_file_path=os.path.join(dataset_dir, geom_csv), sep=',', decimal='.', header = 'infer',
    cols_idx=[1, 3],
    normalize = False
)

c = gimli.array_ptos[:, 0].max() - gimli.array_ptos[:, 0].min()
T, P, rho = GIMLI.Backpack.isa_atmosphere(h=11000)
mu = GIMLI.Backpack.Sutherland_law(mu0 = 1.716e-5, T=T, Treference=255.55)

gimli.compute_param(
    name = 'Re',
    formula = "L * Mach*sqrt(1.4*287*T)*rho/mu",
    externals={
        "rho": rho,
        "mu": mu,
        "L": c,
        "T": T
    }
)

gimli.compute_param(
    name = 'T',
    formula = "T",
    externals={
        "T": T
    }
)

gimli.compute_param(
    name = 'P',
    formula = "P",
    externals={
        "P": P
    }
)

gimli.generate_folders(
    base_files=["journal.jou", "SLURM_FLUENT_2D.job"],
    script_dir="/home/m.jaraiz/repos/CETACEO_UPM/cetaceo/data/f22_fluent/sources/",
    folder_fmt = "aoa_{AoA:.4f}_mach_{Mach:.4f}_h_{h:.0f}",
    overwrite=True,
    update_base_files=True,
    data_to_update={
        "AOA_PLACEHOLDER": "AoA",
        "MACH_PLACEHOLDER": "Mach",
        "TEMP_PLACEHOLDER": "T",
        "P_PLACEHOLDER": "P",
        "FLUENT_TEST": "f22_fluent"
    }
    
)

# asignaciones = gimli.assign_jobs(
#     nodes =[f"n00{n}" for n in [5, 6, 7]],
#     cpus_per_job=48
# )

In [ ]:
gimli.df_cases

In [ ]:
n_cases = gimli.case_tensor.shape[0]
print(f"Creating {n_cases} simulation folders in {gimli.root_dir}")

for i, row in enumerate(gimli.case_tensor):
    
    case_dict = {var: float(row[j]) for j, var in enumerate(gimli.design_vars)}


In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.scatter(x=gimli.df_cases['AoA'], y= gimli.df_cases['Mach'], s=5, c=gimli.df_cases['Re'], cmap='turbo')
plt.colorbar()